In [ ]:
!pip install pandas numpy faiss-cpu sentence-transformers

In [ ]:
import pandas as pd
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer

In [ ]:
# PASTE YOUR CLEANED CSV RAW GITHUB LINK HERE
cleaned_url = "https://raw.githubusercontent.com/Saung210/budgetbites_recipe_chatbot/main/data/cleaned/recipe_dataset_cleaned.csv"

df = pd.read_csv(cleaned_url)

print(df.shape)
df.head()

(483, 20)


,Recipe_name,Category,Cuisine,Preptime,Tottime,Yield,Serving_size,Nutrition_per_serving,Keywords,Description,Recipe_steps,Ingredients,Flag_bad_time,Flag_bad_ingredients,Flag_bad_steps,Calories,Protein_g,Fats_g,Carbs_g,Flag_bad_nutrition
0,Banana Flambe Recipe,Dessert,European,5,10,2 Servings,1 bowl,Calories: 210 kcal Protein: 2 g Fats: 8 g Carb...,"['Banana', 'Flambe', 'Recipe']",A rich and caramelized dessert made with banan...,1.Melt butter in a pan and add sliced bananas....,"Bananas, butter, sugar, cinnamon, rum (optiona...",False,False,False,210,2,8,35,False
1,Greek Cheese Balls Recipe,Snack,European,10,20,8 Servings,1 bowl,Calories: 280 kcal Protein: 10 g Fats: 18 g Ca...,"['Greek', 'Cheese', 'Balls', 'Recipe']","Crispy fried cheese balls with a soft, melty c...","1.Mix cheeses, herbs, and flour into a dough. ...","Feta cheese, mozzarella, flour, eggs, breadcru...",False,False,False,280,10,18,20,False
2,Oreo Cake Recipe,Dessert,European,10,15,5 Servings,1 bowl,Calories: 320 kcal Protein: 5 g Fats: 15 g Car...,"['Oreo', 'Cake', 'Recipe']",A quick and easy chocolate cake made using cru...,1.Crush Oreo biscuits into a fine powder. \n2....,"Oreo biscuits, milk, sugar, baking powder, butter",False,False,False,320,5,15,42,False
3,Grilled Fish in Lemon Butter Sauce Recipe,Lunch,European,10,20,2 Servings,1 bowl,Calories: 260 kcal Protein: 25 g Fats: 14 g Ca...,"['Grilled', 'Fish', 'in', 'Lemon', 'Butter', '...",A light and healthy dish featuring perfectly g...,1.Season fish with salt and pepper. \n2.Grill ...,"Fish fillets, butter, lemon juice, garlic, sal...",False,False,False,260,25,14,5,False
4,Grilled Vegetables Parcels Recipe,Snack,European,10,30,2 Servings,1 bowl,Calories: 180 kcal Protein: 4 g Fats: 9 g Carb...,"['Grilled', 'Vegetables', 'Parcels', 'Recipe']",A healthy and colorful mix of vegetables wrapp...,1.Chop vegetables and season with oil and spic...,"Bell peppers, zucchini, carrots, onions, olive...",False,False,False,180,4,9,22,False


In [ ]:
print(df.columns.tolist())

['Recipe_name', 'Category', 'Cuisine', 'Preptime', 'Tottime', 'Yield', 'Serving_size', 'Nutrition_per_serving', 'Keywords', 'Description', 'Recipe_steps', 'Ingredients', 'Flag_bad_time', 'Flag_bad_ingredients', 'Flag_bad_steps', 'Calories', 'Protein_g', 'Fats_g', 'Carbs_g', 'Flag_bad_nutrition']


In [ ]:
def make_recipe_text(row):
    return f"""
    Recipe Name: {row['Recipe_name']}
    Category: {row['Category']}
    Cuisine: {row['Cuisine']}
    Ingredients: {row['Ingredients']}
    Description: {row['Description']}
    Total Time: {row['Tottime']} minutes
    Calories: {row['Calories']}
    Protein: {row['Protein_g']} g
    Fats: {row['Fats_g']} g
    Carbs: {row['Carbs_g']} g
    """.strip()

df["recipe_text"] = df.apply(make_recipe_text, axis=1)

df[["Recipe_name", "recipe_text"]].head(2)

,Recipe_name,recipe_text
0,Banana Flambe Recipe,Recipe Name: Banana Flambe Recipe\n Categor...
1,Greek Cheese Balls Recipe,Recipe Name: Greek Cheese Balls Recipe\n Ca...


In [ ]:
model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
recipe_texts = df["recipe_text"].tolist()
recipe_embeddings = model.encode(recipe_texts, show_progress_bar=True)
recipe_embeddings = np.array(recipe_embeddings).astype("float32")

print(recipe_embeddings.shape)

Batches:   0%|          | 0/16 [00:00<?, ?it/s]

(483, 384)


In [ ]:
dimension = recipe_embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)
index.add(recipe_embeddings)

print("Number of recipes stored in FAISS:", index.ntotal)

Number of recipes stored in FAISS: 483


In [ ]:
def build_query(ingredients, cuisine=None, nutrition_goal=None, category=None):
    parts = []

    if ingredients:
        parts.append(f"Ingredients: {ingredients}")
    if cuisine and cuisine != "Any":
        parts.append(f"Cuisine: {cuisine}")
    if nutrition_goal and nutrition_goal != "Any":
        parts.append(f"Nutrition Goal: {nutrition_goal}")
    if category and category != "Any":
        parts.append(f"Category: {category}")

    return " | ".join(parts)

In [ ]:
def retrieve_recipes(
    ingredients,
    max_cooking_time=None,
    cuisine=None,
    nutrition_goal=None,
    category=None,
    top_k=5,
    search_k=50
):
    # Build query text
    query = build_query(
        ingredients=ingredients,
        cuisine=cuisine,
        nutrition_goal=nutrition_goal,
        category=category
    )

    # Embed query
    query_embedding = model.encode([query])
    query_embedding = np.array(query_embedding).astype("float32")

    # Search FAISS
    distances, indices = index.search(query_embedding, search_k)
    retrieved_df = df.iloc[indices[0]].copy()

    # Apply hard filters
    if max_cooking_time is not None:
        retrieved_df = retrieved_df[retrieved_df["Tottime"] <= max_cooking_time]

    if cuisine is not None and cuisine != "Any":
        retrieved_df = retrieved_df[
            retrieved_df["Cuisine"].str.strip().str.lower() == cuisine.strip().lower()
        ]

    if category is not None and category != "Any":
        retrieved_df = retrieved_df[
            retrieved_df["Category"].str.strip().str.lower() == category.strip().lower()
        ]

    if retrieved_df.empty:
       return retrieved_df

    # Nutrition ranking
    if nutrition_goal == "High protein":
        retrieved_df = retrieved_df.sort_values(by="Protein_g", ascending=False)
    elif nutrition_goal == "Low calorie":
        retrieved_df = retrieved_df.sort_values(by="Calories", ascending=True)
    elif nutrition_goal == "Low fat":
        retrieved_df = retrieved_df.sort_values(by="Fats_g", ascending=True)

    return retrieved_df.head(top_k)

In [ ]:
results = retrieve_recipes(
    ingredients="chicken, rice, onion",
    max_cooking_time=30,
    cuisine="Indian",
    nutrition_goal="High protein",
    category="Any",
    top_k=5
)

results[[
    "Recipe_name",
    "Cuisine",
    "Category",
    "Tottime",
    "Calories",
    "Protein_g",
    "Fats_g",
    "Carbs_g"
]]

,Recipe_name,Cuisine,Category,Tottime,Calories,Protein_g,Fats_g,Carbs_g
124,Punjabi Mix Dal Recipe,Indian,Lunch,30,290,14,10,38
206,Egg Bonda Recipe,Indian,Snack,30,300,11,18,22
163,Lehsuni Tikki Recipe,Indian,Snack,25,200,6,7,26
9,Corn Pakoda Recipe,Indian,Snack,30,220,6,10,28
164,Corn and Potato Tikki Recipe,Indian,Snack,25,210,5,8,30


In [ ]:
results[[
    "Recipe_name",
    "Description",
    "Ingredients",
    "Recipe_steps"
]].head(1)

,Recipe_name,Description,Ingredients,Recipe_steps
124,Punjabi Mix Dal Recipe,A protein-rich blend of different dals cooked ...,"Mixed lentils, onions, tomatoes, spices",1.Cook mixed lentils. \n2.Prepare tempering. \...


In [ ]:
faiss.write_index(index, "recipe_faiss.index")
df.to_csv("recipe_dataset_with_text.csv", index=False)

print("Saved recipe_faiss.index and recipe_dataset_with_text.csv")

Saved recipe_faiss.index and recipe_dataset_with_text.csv
